# 02 — Fine-Tune NaijaReviewer-8B (GPU runtime, Unsloth)

**Run this on a GPU runtime** (Runtime → Change runtime type → A100 / L4 / T4).

**Prerequisite**: `01_build_corpus.ipynb` has been run and produced `v1_train_alpaca.jsonl` etc. on Drive.

## What this notebook does
1. Loads the Alpaca-formatted corpus from Drive
2. QLoRA fine-tunes Llama 3.1 8B Instruct via Unsloth (2.4× faster than vanilla TRL)
3. Resumes from `checkpoint-XXXX` on Drive if interrupted
4. Merges LoRA into base + smoke-tests inference
5. Pushes merged model + adapter + corpus dataset to HuggingFace

## Cost
- Colab Pro+ A100: ~1.5–2 hours of GPU time
- L4: ~3 hours
- T4: ~5–6 hours (still works thanks to Unsloth's VRAM savings)

## Inputs needed
- `HF_TOKEN` — for Llama 3.1 8B download + model/dataset push
- `ANTHROPIC_API_KEY` — optional (used only for eval baseline)

Accept the Llama 3.1 license first: [meta-llama/Meta-Llama-3.1-8B-Instruct](https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct)

## 0. Setup — Unsloth Install (no data_designer here)

In [ ]:
# Unsloth-only install — clean stack, no conflict with corpus-build deps
!pip install -q --upgrade pip
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.13" peft accelerate bitsandbytes wandb sentencepiece
!pip install -q pandas jsonlines pyarrow datasets

import torch
from unsloth import FastLanguageModel
print(f"\n✅ torch {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print("✅ unsloth ready")

In [ ]:
# Drive mount + paths
import os, sys, json
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_DIR = Path("/content/drive/MyDrive/naija_persona_corpus")
else:
    DRIVE_DIR = Path.home() / "naija_persona_corpus-runs"

CORPUS_DIR    = DRIVE_DIR / "processed"
TRAINING_CKPT = DRIVE_DIR / "checkpoints" / "naija-reviewer-v1"
MERGED_DIR    = DRIVE_DIR / "merged" / "naija-reviewer-8b"
RESULTS_DIR   = DRIVE_DIR / "results"
for d in (TRAINING_CKPT, MERGED_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Verify corpus exists
train_path = CORPUS_DIR / "v1_train_alpaca.jsonl"
if not train_path.exists():
    raise RuntimeError(
        f"\n❌ Corpus not found at {train_path}\n"
        f"   Run 01_build_corpus.ipynb first on a CPU runtime, then return here."
    )
n_train = sum(1 for _ in train_path.open())
print(f"✅ Drive: {DRIVE_DIR}")
print(f"✅ Corpus found: {n_train:,} training rows")

In [ ]:
# Secrets — only HF + (optional) Anthropic
def _set_secret(name, prompt, required=True):
    if IN_COLAB:
        from google.colab import userdata
        try:
            v = userdata.get(name)
            if v:
                os.environ[name] = v
                print(f"  ✅ {name} from Colab Secrets")
                return v
        except Exception: pass
    v = os.environ.get(name)
    if not v and required:
        from getpass import getpass
        v = getpass(prompt); os.environ[name] = v
    if v: print(f"  ✅ {name} configured")
    elif required: raise RuntimeError(f"{name} required")
    return v

_set_secret("HF_TOKEN", "HF_TOKEN (hf_…): ", required=True)
_set_secret("ANTHROPIC_API_KEY", "ANTHROPIC_API_KEY (optional, for eval): ", required=False)

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

## 1. GPU Autotune

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("No GPU — Runtime → Change runtime type → A100/L4/H100/T4")

GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9
BF16_OK  = torch.cuda.is_bf16_supported()

name_l = GPU_NAME.lower()
if   "h100" in name_l:                  CFG = {"batch":16,"grad_accum":2, "workers":4,"seq_len":4096}
elif "a100" in name_l and GPU_VRAM>60:  CFG = {"batch":16,"grad_accum":2, "workers":4,"seq_len":4096}
elif "a100" in name_l:                  CFG = {"batch":8, "grad_accum":4, "workers":4,"seq_len":4096}
elif "l4"   in name_l:                  CFG = {"batch":4, "grad_accum":8, "workers":2,"seq_len":3072}
elif "t4"   in name_l:                  CFG = {"batch":2, "grad_accum":16,"workers":2,"seq_len":2048}
else:                                   CFG = {"batch":4, "grad_accum":8, "workers":2,"seq_len":3072}

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True

print(f"GPU: {GPU_NAME} ({GPU_VRAM:.1f}GB)  bf16={BF16_OK}")
print(f"Autotuned: batch={CFG['batch']} × grad_accum={CFG['grad_accum']} = {CFG['batch']*CFG['grad_accum']} effective")
print(f"           seq_len={CFG['seq_len']}, workers={CFG['workers']}")

## 2. Load Model + Dataset + Configure Trainer

In [ ]:
# Resume detection
checkpoints = sorted(TRAINING_CKPT.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))
RESUME_FROM = checkpoints[-1] if checkpoints else None
print(f"resume: {RESUME_FROM.name if RESUME_FROM else 'fresh start'}")

In [ ]:
# Load Llama 3.1 8B via Unsloth
from unsloth import FastLanguageModel

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

if RESUME_FROM:
    print(f"Loading from checkpoint {RESUME_FROM.name}...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(RESUME_FROM),
        max_seq_length=CFG["seq_len"], dtype=None, load_in_4bit=True,
    )
    FastLanguageModel.for_training(model)
else:
    print("Loading Llama 3.1 8B (Unsloth pre-quantized 4-bit, ~5GB)...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
        max_seq_length=CFG["seq_len"], dtype=None, load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model, r=LORA_R, target_modules=LORA_TARGETS,
        lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        bias="none", use_gradient_checkpointing="unsloth", random_state=42,
    )

tokenizer.padding_side = "right"
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n✅ trainable: {trainable/1e6:.1f}M params")

In [ ]:
# Load Alpaca data from Drive (produced by 01_build_corpus.ipynb)
from datasets import Dataset

def alpaca_to_text(r):
    return {"text": f"### Instruction\n{r['instruction']}\n\n### Input\n{r['input']}\n\n### Response\n{r['output']}"}

ds_train_raw = [json.loads(l) for l in (CORPUS_DIR / "v1_train_alpaca.jsonl").open()]
ds_val_raw   = [json.loads(l) for l in (CORPUS_DIR / "v1_val_alpaca.jsonl").open()]
ds_train = Dataset.from_list(ds_train_raw).map(alpaca_to_text)
ds_val   = Dataset.from_list(ds_val_raw).map(alpaca_to_text)
print(f"train={len(ds_train):,}  val={len(ds_val):,}")
print(f"\nSample:\n{ds_train[0]['text'][:500]}")

In [ ]:
# W&B (optional)
import wandb
try:
    wandb.init(project="naija_persona_corpus", name="naija-reviewer-8b-v1",
               config={"gpu": GPU_NAME, "vram": GPU_VRAM, **CFG},
               mode="online", resume="allow")
    print("✅ W&B online")
except Exception as e:
    print(f"⚠️ W&B disabled: {e}")
    os.environ["WANDB_DISABLED"] = "true"

## 3. Train

In [ ]:
from trl import SFTTrainer, SFTConfig
import time

training_args = SFTConfig(
    output_dir=str(TRAINING_CKPT),
    num_train_epochs=2,
    per_device_train_batch_size=CFG["batch"],
    per_device_eval_batch_size=CFG["batch"],
    gradient_accumulation_steps=CFG["grad_accum"],
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    weight_decay=0.01,
    bf16=BF16_OK,
    fp16=not BF16_OK,
    optim="paged_adamw_8bit",
    logging_steps=25,
    save_strategy="steps",
    save_steps=200,
    eval_strategy="steps",
    eval_steps=200,
    save_total_limit=3,
    seed=42,
    report_to=["wandb"] if not os.environ.get("WANDB_DISABLED") else ["none"],
    run_name="naija-reviewer-8b-v1",
    max_length=CFG["seq_len"],
    packing=False,
    dataset_text_field="text",
    save_safetensors=True,
    tf32=True,
    group_by_length=True,
)
trainer = SFTTrainer(
    model=model, train_dataset=ds_train, eval_dataset=ds_val,
    tokenizer=tokenizer, args=training_args,
)

print("🚀 training...")
t0 = time.time()
if RESUME_FROM:
    trainer.train(resume_from_checkpoint=str(RESUME_FROM))
else:
    trainer.train()
print(f"\n✅ done in {(time.time()-t0)/60:.1f}min")

FINAL_ADAPTER = TRAINING_CKPT / "final-adapter"
model.save_pretrained(str(FINAL_ADAPTER))
tokenizer.save_pretrained(str(FINAL_ADAPTER))
print(f"✅ adapter → {FINAL_ADAPTER}")

In [ ]:
# Merge LoRA into base (Unsloth one-liner)
import gc
if (MERGED_DIR / "config.json").exists():
    print("✅ merged exists — skipping")
else:
    model.save_pretrained_merged(str(MERGED_DIR), tokenizer, save_method="merged_16bit")
    print(f"✅ merged → {MERGED_DIR}")
    del trainer, model; gc.collect(); torch.cuda.empty_cache()

## 4. Smoke Test

In [ ]:
from unsloth import FastLanguageModel
naija_model, naija_tok = FastLanguageModel.from_pretrained(
    model_name=str(MERGED_DIR), max_seq_length=CFG["seq_len"], dtype=None, load_in_4bit=False,
)
FastLanguageModel.for_inference(naija_model)

ds_test_raw = [json.loads(l) for l in (CORPUS_DIR / "v1_test_alpaca.jsonl").open()]
for i in range(3):
    s = ds_test_raw[i]
    prompt = f"### Instruction\n{s['instruction']}\n\n### Input\n{s['input']}\n\n### Response\n"
    inputs = naija_tok(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = naija_model.generate(**inputs, max_new_tokens=200, temperature=0.7,
                                    do_sample=True, pad_token_id=naija_tok.eos_token_id)
    result = naija_tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    print(f"\n=== Sample {i+1} ===")
    print(f"GT:    {s['output']}")
    print(f"MODEL: {result[:400]}")

## 5. Push to HuggingFace

In [ ]:
from huggingface_hub import create_repo, upload_folder

HF_ORG = "Mystique1337"  # ← change to your org/username
MODEL_REPO   = f"{HF_ORG}/naija-reviewer-8b"
ADAPTER_REPO = f"{HF_ORG}/naija-reviewer-8b-lora"
DATASET_REPO = f"{HF_ORG}/npa-corpus-v1"

for repo, kind in [(MODEL_REPO,"model"), (ADAPTER_REPO,"model"), (DATASET_REPO,"dataset")]:
    try: create_repo(repo, repo_type=kind, exist_ok=True, private=True); print(f"  ✅ {repo}")
    except Exception as e: print(f"  ⚠️ {repo}: {e}")

print(f"\nUploading merged model...")
upload_folder(folder_path=str(MERGED_DIR), repo_id=MODEL_REPO, repo_type="model")
print(f"  ✅ https://huggingface.co/{MODEL_REPO}")

print(f"\nUploading adapter...")
upload_folder(folder_path=str(FINAL_ADAPTER), repo_id=ADAPTER_REPO, repo_type="model",
              ignore_patterns=["merged/*", "checkpoint-*", "runs/*"])
print(f"  ✅ https://huggingface.co/{ADAPTER_REPO}")

print(f"\nUploading corpus...")
upload_folder(folder_path=str(CORPUS_DIR), repo_id=DATASET_REPO, repo_type="dataset",
              ignore_patterns=["raw_*", "checkpoints/*"])
print(f"  ✅ https://huggingface.co/datasets/{DATASET_REPO}")
print("\n🎉 done (private — flip to public when ready)")

## 6. GGUF Conversion for Ollama (local step)

On your laptop:

```bash
git clone https://github.com/ggerganov/llama.cpp && cd llama.cpp && make
huggingface-cli download Mystique1337/naija-reviewer-8b --local-dir ./merged
python3 convert_hf_to_gguf.py ./merged --outfile naija-reviewer-8b-f16.gguf --outtype f16
./llama-quantize naija-reviewer-8b-f16.gguf naija-reviewer-8b-Q4_K_M.gguf Q4_K_M
cp naija-reviewer-8b-Q4_K_M.gguf telcoproject/finetuning/
cd telcoproject && ollama create naija-reviewer-8b -f finetuning/Modelfile
```

Then in `.env`:
```
TASK1_BACKBONE=ollama:naija-reviewer-8b
```

Restart `make demo` — FastAPI now serves the local fine-tune at zero API cost.